# Классификация текстов с применением RNN

## Загрузка данных

In [1]:
import pandas as pd

df = pd.read_csv('Petitions.csv', nrows=30000)
df

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство
...,...,...,...
29995,3346982,"Мусор на газоне, мусор и песок на пешеходной д...",Благоустройство
29996,3035580,Сломана опора лесенки на детской площадке.\nНе...,Благоустройство
29997,3303664,Добрый день!\r\nПрошу проверить тротуары на ул...,Благоустройство
29998,3174526,В январе 2021 года управляющей организацией (д...,Содержание МКД


Видим, что датасет - обращения жильцов в ЖКХ. Все обращения на русском языке.

## Предобработка

Напишем функция для предобработки данных, которая будет включать:
1. Очистку: приведем все к нижнему регистру, заменим все числовые вхождения на токен NUM т.к. смысловой информации числа не будут нести и на всякий случай оставим только русские символы (вдруг где-то опечатка и имеется не кириллица);
2. Токенизация при помощи библиотеки `razdel`, разработанной специально для русского языка;
3. Лемматизацию, т.к. дальше для векторизации будет использована специально обученная модель для русского языка, также удаление стоп-слов.

In [2]:
import re
import pymorphy3
from razdel import tokenize
import nltk
from nltk.corpus import stopwords


# Инициализация инструментов
nltk.download('stopwords')
stop_words = set(stopwords.words('russian'))
morph = pymorphy3.MorphAnalyzer()


def preprocess_text(text):
    # 1. Очистка: нижний регистр и замена чисел на NUM
    text = str(text).lower()
    text = re.sub(r'\d+', ' num ', text)
    
    # Оставляем только кириллицу и наш токен num
    text = re.sub(r'[^а-яё\snum]', ' ', text)
    
    # 2. Токенизация при помощи razdel
    tokens = [_.text for _ in tokenize(text)]
    
    # 3. Лемматизация и удаление стоп-слов
    clean_tokens = []
    for word in tokens:
        # Если в слове есть латиница, но это не 'num' — игнорируем его
        if re.search(r'[a-z]', word) and word != 'num':
            continue
        # Получаем нормальную форму слова (лемму)
        lemma = morph.parse(word)[0].normal_form
        if (lemma not in stop_words or lemma == 'num') and len(lemma) > 1:
            clean_tokens.append(lemma)
            
    return clean_tokens

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [3]:
df['preprocessed_text'] = df['public_petition_text'].apply(preprocess_text)
df

,id,public_petition_text,reason_category,preprocessed_text
0,3168490,снег на дороге,Благоустройство,"[снег, дорога]"
1,3219678,очистить кабельный киоск от рекламы,Благоустройство,"[очистить, кабельный, киоск, реклама]"
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство,"[просить, убрать, всё, дерево, кустарник, кото..."
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД,"[неудовлетворительный, состояние, парадный, на..."
4,3336285,Граффити,Благоустройство,[граффити]
...,...,...,...,...
29995,3346982,"Мусор на газоне, мусор и песок на пешеходной д...",Благоустройство,"[мусор, газон, мусор, песок, пешеходный, дорож..."
29996,3035580,Сломана опора лесенки на детской площадке.\nНе...,Благоустройство,"[сломать, опора, лесенка, детский, площадка, н..."
29997,3303664,Добрый день!\r\nПрошу проверить тротуары на ул...,Благоустройство,"[добрый, день, просить, проверить, тротуар, ул..."
29998,3174526,В январе 2021 года управляющей организацией (д...,Содержание МКД,"[январь, num, год, управлять, организация, дал..."


Видим, что функция предобработки сработала прекрасно, все числа заменены на токен 'num', все слова приведены в начальную форму, удалены стоп-слова.

## Кодирование категориального признака

Признак `reason_category` - категориальный, нам это не подходит, закодируем категории в числа

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['target'] = le.fit_transform(df['reason_category'])

mapping = pd.DataFrame({'category': le.classes_, 'encoded': range(len(le.classes_))})
mapping

,category,encoded
0,Благоустройство,0
1,Водоотведение,1
2,Водоснабжение,2
3,Кровля,3
4,Нарушение порядка пользования общим имуществом,4
5,Нарушение правил пользования общим имуществом,5
6,Незаконная информационная и (или) рекламная ко...,6
7,Незаконная реализация товаров с торгового обор...,7
8,Повреждения или неисправность элементов улично...,8
9,Подвалы,9


In [5]:
df

,id,public_petition_text,reason_category,preprocessed_text,target
0,3168490,снег на дороге,Благоустройство,"[снег, дорога]",0
1,3219678,очистить кабельный киоск от рекламы,Благоустройство,"[очистить, кабельный, киоск, реклама]",0
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство,"[просить, убрать, всё, дерево, кустарник, кото...",0
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД,"[неудовлетворительный, состояние, парадный, на...",11
4,3336285,Граффити,Благоустройство,[граффити],0
...,...,...,...,...,...
29995,3346982,"Мусор на газоне, мусор и песок на пешеходной д...",Благоустройство,"[мусор, газон, мусор, песок, пешеходный, дорож...",0
29996,3035580,Сломана опора лесенки на детской площадке.\nНе...,Благоустройство,"[сломать, опора, лесенка, детский, площадка, н...",0
29997,3303664,Добрый день!\r\nПрошу проверить тротуары на ул...,Благоустройство,"[добрый, день, просить, проверить, тротуар, ул...",0
29998,3174526,В январе 2021 года управляющей организацией (д...,Содержание МКД,"[январь, num, год, управлять, организация, дал...",11


## Векторизация

Для преобразования текстовых данных в числовые векторы используется библиотека Navec, являющаяся частью проекта Natasha. В данной работе применяется модель `navec_hudlit_v1_12B_500K_300d_100q`, обученная на корпусе художественной литературы.

In [6]:
from navec import Navec

navec_path = 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(navec_path)

На данном этапе токены преобразуются в числовые векторы с использованием весов Navec, и формируется единый тензор данных.

In [7]:
import torch
from torch.nn.utils.rnn import pad_sequence

all_sequences = []
for tokens in df['preprocessed_text']:
    # Для каждого слова берем вектор из Navec, если слова нет — вектор <unk>
    word_vectors = [torch.tensor(navec[w]) if w in navec else torch.tensor(navec['<unk>']) for w in tokens]
    
    # Если после очистки текст пустой, добавим один нулевой вектор
    if not word_vectors:
        word_vectors = [torch.zeros(300)]
        
    all_sequences.append(torch.stack(word_vectors))

# 3. Делаем Padding: уравниваем длины всех текстов
# Результат: тензор [Количество_текстов, Макс_длина, 300]
X = pad_sequence(all_sequences, batch_first=True, padding_value=0.0).float()
y = torch.tensor(df['target'].values).long()

## Разделение данных на выборки

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [10]:
batch_size = 128

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Нейронная сеть

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Для проведения сравнительного анализа была реализована модель RNNClassifier, поддерживающая три типа стандартных слоев: простую рекуррентную сеть (RNN), сеть с долгой краткосрочной памятью (LSTM) и управляемые рекуррентные блоки (GRU).

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class RNNClassifier(nn.Module):
    def __init__(self, nn_type, input_size, hidden_size, num_classes):
        super().__init__()
        self.nn_type = nn_type
        self.rnn_layer = self._get_layer(nn_type, input_size, hidden_size)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def _get_layer(self, nn_type, input_size, hidden_size):
        if nn_type == 'RNN':
            return nn.RNN(input_size, hidden_size, batch_first=True)
        elif nn_type == 'LSTM':
            return nn.LSTM(input_size, hidden_size, batch_first=True)
        elif nn_type == 'GRU':
            return nn.GRU(input_size, hidden_size, batch_first=True)
        else:
            raise ValueError("RNN, LSTM, GRU")

    def forward(self, x):
        # x: [batch, seq_len, 300]
        out, _ = self.rnn_layer(x)
        
        # Извлекаем последний скрытый слой (Many-to-One)
        # Если это LSTM, out — это тензор, всё ок
        last_step = out[:, -1, :] 
        
        # Стабилизируем выход
        last_step = self.bn(last_step)
        last_step = self.dropout(last_step)
        
        return self.fc(last_step)

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight('balanced', classes=np.unique(y_train.cpu().numpy()), y=y_train.cpu().numpy())
# Смягчаем веса (берем квадратный корень), чтобы не было гигантских разрывов
weights = np.sqrt(weights) 
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

Для обучения всех моделей была разработана универсальная функция train_and_evaluate_model. Она инкапсулирует в себе полный цикл жизни нейросети: от инициализации до формирования отчета о качестве классификации.

In [14]:
from sklearn.metrics import classification_report
from torch.nn.utils import clip_grad_norm_
import torch.optim as optim

def train_and_evaluate_model(train_loader, test_loader, input_size=300, hidden_size=128, num_classes=None, epochs=50, model=None, nn_type=None, lr=0.001):
    # 1. Логика выбора модели
    if model is None:
        # Если готовую модель не дали, создаем стандартную (RNN/LSTM/GRU)
        model = RNNClassifier(nn_type, input_size, hidden_size, num_classes).to(device)
    else:
        # Если модель передана (наш кастом), используем её
        model = model.to(device)
        if nn_type is None:
            nn_type = model.__class__.__name__

    # 2. Настройка обучения
    criterion = nn.CrossEntropyLoss() 
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Можно добавить планировщик, чтобы при плато на Loss снижать шаг обучения
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5, threshold=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Клиппинг для стабильности рекуррентных слоев
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        scheduler.step(avg_loss) # Обновляем планировщик
        
        if (epoch + 1) % 1 == 0:
            print(f"{nn_type} | Эпоха {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

    # 3. Оценка (Evaluation)
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            out = model(inputs)
            all_preds.extend(out.argmax(dim=-1).cpu().numpy())
            all_true.extend(targets.numpy())
    
    print(f"\nОтчет для {nn_type}:")
    report = classification_report(all_true, all_preds, labels=range(num_classes), target_names=le.classes_, zero_division=0)
    print(report)
    return report # Возвращаем отчет для сравнения

In [15]:
INPUT_SIZE = 300 # Вектор Navec
HIDDEN_SIZE = 128
NUM_CLASSES = len(le.classes_)

# Словарь для хранения обученных моделей, если понадобятся позже
results = {}

for name in ['RNN', 'LSTM', 'GRU']:
    print(f"\nЗапуск обучения для: {name}")
    results[name] = train_and_evaluate_model(
        train_loader=train_dataloader,
        test_loader=test_dataloader,
        nn_type=name,
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_classes=NUM_CLASSES,
        epochs=40
    )


Запуск обучения для: RNN
RNN | Эпоха 1/40 | Loss: 1.7318
RNN | Эпоха 2/40 | Loss: 1.3742
RNN | Эпоха 3/40 | Loss: 1.5086
RNN | Эпоха 4/40 | Loss: 1.4499
RNN | Эпоха 5/40 | Loss: 1.4320
RNN | Эпоха 6/40 | Loss: 1.4196
RNN | Эпоха 7/40 | Loss: 1.4098
RNN | Эпоха 8/40 | Loss: 1.4058
RNN | Эпоха 9/40 | Loss: 1.4073
RNN | Эпоха 10/40 | Loss: 1.4026
RNN | Эпоха 11/40 | Loss: 1.3995
RNN | Эпоха 12/40 | Loss: 1.3991
RNN | Эпоха 13/40 | Loss: 1.3996
RNN | Эпоха 14/40 | Loss: 1.3980
RNN | Эпоха 15/40 | Loss: 1.3977
RNN | Эпоха 16/40 | Loss: 1.3947
RNN | Эпоха 17/40 | Loss: 1.3947
RNN | Эпоха 18/40 | Loss: 1.3933
RNN | Эпоха 19/40 | Loss: 1.3910
RNN | Эпоха 20/40 | Loss: 1.3922
RNN | Эпоха 21/40 | Loss: 1.3916
RNN | Эпоха 22/40 | Loss: 1.3939
RNN | Эпоха 23/40 | Loss: 1.3929
RNN | Эпоха 24/40 | Loss: 1.3920
RNN | Эпоха 25/40 | Loss: 1.3935
RNN | Эпоха 26/40 | Loss: 1.3917
RNN | Эпоха 27/40 | Loss: 1.3908
RNN | Эпоха 28/40 | Loss: 1.3914
RNN | Эпоха 29/40 | Loss: 1.3910
RNN | Эпоха 30/40 | Loss: 

In [16]:
class MyGRUBlock(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(MyGRUBlock, self).__init__()
        # input_size: размер входного вектора — 300
        # hidden_size: размер внутреннего состояния — 128
        self.hidden_size = hidden_size
        
        # x_gates: веса для обработки входящего слова (xt). 
        # Умножаем на 3, чтобы одним слоем рассчитать заготовки для z, r и h_tilde
        self.x_gates = nn.Linear(input_size, 3 * hidden_size)
        
        # h_gates: веса для обработки предыдущей памяти (ht_prev).
        # Также рассчитаны на 3 гейта сразу
        self.h_gates = nn.Linear(hidden_size, 3 * hidden_size)
        
        self.reset_parameters()

    def reset_parameters(self):
        # Инициализация весов Xavier для стабильного старта
        nn.init.xavier_uniform_(self.x_gates.weight)
        nn.init.xavier_uniform_(self.h_gates.weight)
        nn.init.zeros_(self.x_gates.bias)
        nn.init.zeros_(self.h_gates.bias)
        
        with torch.no_grad():
            # Заполняем bias для Update Gate (z) единицами, 
            # чтобы в начале обучения модель "по умолчанию" ничего не забывала
            self.x_gates.bias[:self.hidden_size].fill_(1.0)
            self.h_gates.bias[:self.hidden_size].fill_(1.0)

    def forward(self, xt, ht_prev):
        # gates_x: результат умножения текущего слова на веса (W * xt)
        # gates_h: результат умножения прошлой памяти на веса (U * ht_prev)
        gates_x = self.x_gates(xt)
        gates_h = self.h_gates(ht_prev)
        
        # chunk(3, 1) разрезает большие тензоры на 3 части по 128 элементов:
        # x_z, h_z: части для Update Gate (вентиль обновления)
        # x_r, h_r: части для Reset Gate (вентиль сброса)
        # x_h, h_h: части для вычисления Кандидата (нового смысла)
        x_z, x_r, x_h = gates_x.chunk(3, 1)
        h_z, h_r, h_h = gates_h.chunk(3, 1)
        
        # zt (Update Gate): определяет, сколько старой памяти (ht_prev) пропустить дальше
        zt = torch.sigmoid(x_z + h_z)
        
        # rt (Reset Gate): определяет, какую часть прошлого (ht_prev) нужно сбросить (забыть),
        # прежде чем анализировать текущее слово
        rt = torch.sigmoid(x_r + h_r)
        
        # ht_tilde (Candidate): "новое предложение" по памяти. 
        # Здесь rt (reset) напрямую умножается на h_h, обнуляя или сохраняя прошлое
        ht_tilde = torch.tanh(x_h + rt * h_h)

        # ht (Final Hidden State): финальный микс старого и нового.
        # Если zt = 1, берем больше нового. Если zt = 0, оставляем старое.
        ht = (1 - zt) * ht_prev + zt * ht_tilde
        
        return ht

In [17]:
class MyGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_p=0.2):
        super(MyGRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.gru_block = MyGRUBlock(input_size, hidden_size)
        
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        batch_size = x.size(0)
        seq_len = x.size(1)
        device = x.device
        
        ht = torch.zeros(batch_size, self.hidden_size).to(device)

        for i in range(seq_len):
            xt = x[:, i, :] 
            ht = self.gru_block(xt, ht)
        out = self.dropout(ht)
        return self.fc(out)

In [18]:
my_custom_model = MyGRUModel(input_size=300, hidden_size=128, output_size=len(le.classes_))

print(f"\nЗапуск обучения для: CustomGRU")
results['CustomGRU'] = train_and_evaluate_model(
    train_dataloader, test_dataloader, 
    num_classes=len(le.classes_), 
    epochs=20, model=my_custom_model, lr=0.005
)


Запуск обучения для: CustomGRU
MyGRUModel | Эпоха 1/20 | Loss: 1.4469
MyGRUModel | Эпоха 2/20 | Loss: 0.7618
MyGRUModel | Эпоха 3/20 | Loss: 0.3346
MyGRUModel | Эпоха 4/20 | Loss: 0.2516
MyGRUModel | Эпоха 5/20 | Loss: 0.1917
MyGRUModel | Эпоха 6/20 | Loss: 0.1601
MyGRUModel | Эпоха 7/20 | Loss: 0.1338
MyGRUModel | Эпоха 8/20 | Loss: 0.1254
MyGRUModel | Эпоха 9/20 | Loss: 0.1114
MyGRUModel | Эпоха 10/20 | Loss: 0.0995
MyGRUModel | Эпоха 11/20 | Loss: 0.0978
MyGRUModel | Эпоха 12/20 | Loss: 0.1058
MyGRUModel | Эпоха 13/20 | Loss: 0.1067
MyGRUModel | Эпоха 14/20 | Loss: 0.0909
MyGRUModel | Эпоха 15/20 | Loss: 0.0903
MyGRUModel | Эпоха 16/20 | Loss: 0.0957
MyGRUModel | Эпоха 17/20 | Loss: 0.0983
MyGRUModel | Эпоха 18/20 | Loss: 0.0816
MyGRUModel | Эпоха 19/20 | Loss: 0.0818
MyGRUModel | Эпоха 20/20 | Loss: 0.0838

Отчет для MyGRUModel:
                                                                                  precision    recall  f1-score   support

                               